In [1]:
import sys
sys.path.append("..")  # pour importer le dossier sources/, situé à la racine du dépôt

In [2]:
# Chargement des 8 sessions : téléchargées une seule fois, puis relues depuis data/raw/
from sources.chargement_data import charger_parcoursup, charger_poursuite, charger_ips, DATASETS
from sources.cleaning_data import preparer_base, harmoniser_ips, preparer_poursuite, base_selective
from sources.jointures import joindre_ips, panel_academies
from sources.indicateurs import taux_poursuite, part_poursuite_autre_academie_idf, part_admis_meme_academie

dfs = {annee: charger_parcoursup(annee) for annee in DATASETS}

Préparation de la base : identifiant de formation suivi dans le temps (`code`), sélectivité (`selective`) et filière (`filiere`) harmonisées, puis empilement des sessions. Les choix sont justifiés dans `notebooks/exploration.ipynb`.

In [3]:
base = preparer_base(dfs)

print(base.shape)
print("sélectivité manquante :", base["selective"].isna().sum())
base.groupby("session").size()

(104274, 133)
sélectivité manquante : 634


session
2018    10697
2019    11577
2020    12760
2021    13396
2022    13644
2023    13869
2024    14079
2025    14252
dtype: int64

## Enrichissement 1 : l'indice de position sociale (IPS) des lycées

L'IPS, publié par l'Éducation nationale, résume le milieu social des élèves d'un lycée : plus il est élevé, plus le milieu est favorisé. Beaucoup de CPGE et de BTS sont installés dans des lycées. On peut donc rattacher à chaque formation l'IPS du lycée qui l'accueille, grâce au code UAI et à l'année. Les trois fichiers publiés (2016-2021, 2022, 2023-2025) n'ont pas les mêmes formats : on les harmonise avant la jointure.

In [4]:
ips = harmoniser_ips(charger_ips())
base = joindre_ips(base, ips)

# Part des formations qui trouvent l'IPS de leur lycée, par filière (en %)
avec_ips = base["ips_lycee"].notna()
(avec_ips.groupby(base["filiere"]).mean() * 100).round(1).sort_values(ascending=False)

filiere
CPGE                   94.4
BTS                    85.5
Autre                  56.1
Ecole d'ingénieur       4.6
Sanitaire et social     1.0
IUT                     0.1
Ecole de commerce       0.0
LAS                     0.0
Licence                 0.0
Santé                   0.0
Name: ips_lycee, dtype: float64

94 % des CPGE et 86 % des BTS trouvent l'IPS de leur lycée. Les universités, les IUT et les écoles ne sont pas des lycées : la jointure ne les concerne pas. L'analyse fondée sur l'IPS portera donc sur les CPGE et les BTS.

## Enrichissement 2 : le taux de poursuite des néo-bacheliers dans le supérieur (2010-2025)

Ce fichier du ministère donne, pour chaque académie où le bac a été obtenu, chaque année et chaque voie du bac, le nombre de bacheliers et le nombre de ceux qui poursuivent dans le supérieur. Il commence en 2010 : on peut comparer la période APB (jusqu'en 2017) à la période Parcoursup (depuis 2018).

In [5]:
poursuite = preparer_poursuite(charger_poursuite())

# Taux de poursuite (en %) par année et par voie du bac : une colonne par voie
taux_poursuite(poursuite, ["annee", "voie"]).unstack()

voie,Bac général,Bac professionnel,Bac technologique
annee,,,
2010,94.5,34.8,80.5
2011,93.6,34.5,80.5
2012,93.4,35.1,80.4
2013,93.2,39.5,77.2
2014,93.3,35.7,78.8
2015,93.3,36.5,80.7
2016,92.3,36.1,79.4
2017,91.8,38.6,79.0
2018,92.8,39.5,78.7


Le taux de poursuite des bacheliers professionnels passe de 34,8 % en 2010 à 38,6 % en 2017, puis à 46,5 % en 2025. Celui des bacheliers généraux (environ 93 %) et technologiques (environ 80 %) reste stable.

On relie ensuite ce fichier aux admis de Parcoursup, par académie et par année. On obtient un tableau avec une ligne par académie et par année : c'est la jointure entre les deux sources. Attention : Parcoursup donne l'académie de la formation, et le fichier de poursuite l'académie du bac.

In [6]:
panel = panel_academies(base, poursuite)

print(panel.shape, "-", panel["academie"].nunique(), "académies")
panel.head()

(240, 13) - 30 académies


,academie,annee,acc_tot,acc_neobac,acc_bp,acc_bt,acc_brs,bacheliers_Bac général,bacheliers_Bac professionnel,bacheliers_Bac technologique,poursuivants_Bac général,poursuivants_Bac professionnel,poursuivants_Bac technologique
0,Aix-Marseille,2018,24485,18607,2068,3850,4363,15646,7291,6172,14346.33570,2568.58960,4569.22180
1,Aix-Marseille,2019,27080,19253,2287,4081,4614,15268,7063,6176,14116.18750,2891.03480,4783.35260
2,Aix-Marseille,2020,28793,21222,2484,4331,6454,17001,7636,6767,15693.57670,3437.52100,4947.35940
3,Aix-Marseille,2021,27614,19604,2143,3542,5888,16450,7179,5887,15659.18680,3515.97980,4461.22740
4,Aix-Marseille,2022,26776,18561,1911,3455,5708,16159,6812,5649,15171.71334,3293.76365,4450.19656


## Zoom Île-de-France : le secteur unique de 2019

Depuis la session 2019, les académies de Paris, Créteil et Versailles forment un seul secteur dans Parcoursup. On regarde l'effet de ce changement dans les deux sources :
- Parcoursup : la part des admis venant de l'académie de Paris, dans les formations parisiennes ;
- poursuite d'études : la part des bacheliers franciliens qui poursuivent dans une autre académie d'Île-de-France.

In [7]:
import pandas as pd

# Les deux indicateurs côte à côte, en %, de 2016 à 2025 (Parcoursup commence en 2018)
idf = pd.DataFrame({
    "admis de l'académie de Paris dans les formations parisiennes": part_admis_meme_academie(base, "Paris"),
    "bacheliers franciliens poursuivant dans une autre académie d'IDF": part_poursuite_autre_academie_idf(poursuite),
})
idf.loc[2016:]

,admis de l'académie de Paris dans les formations parisiennes,bacheliers franciliens poursuivant dans une autre académie d'IDF
2016,NaN,31.1
2017,NaN,30.6
2018,25.1,31.6
2019,20.8,35.2
2020,19.9,36.1
2021,18.9,37.6
2022,17.8,37.9
2023,18.0,37.8
2024,17.7,37.9
2025,16.4,38.0


ON NE REGARDE QUE LES FORMATIONS SELECTIVES AFIN D'OBSERVER LA DEMOCRATISATION

In [8]:
parcoursup_2018_2025 = pd.read_csv("../data/parcoursup_2018_2025.csv",sep=";")
formations_selectives_2018_2025 = base_selective(parcoursup_2018_2025)

FileNotFoundError: [Errno 2] No such file or directory: '../data/parcoursup_2018_2025.csv'

On obtient une base de données nommée "formations_selectives_2018_2025" contenant 132 paramètres sur les formations séléctives entre 2018 et 2025. 

Nous avons choisi de ne garder que les formations séléctives car nous allons étudier l'évolution des paramètres de séléction des établissements d'enseignement supérieur. Ainsi les données relatives aux établissements non-sélectifs (dont les facultés qui recrutent principalement sur leur secteur géographique) ne paraient pas pertinent. 

Le but est maintenant de regarder l'évolution de ses différents paramètres au fur et à mesure des années afin d'observer ou non des signes d'une démocratisation de l'enseignement supérieur.

In [ ]:

tab1 = tableau_comp_1(formations_selectives_2018_2025, "acc_brs", "acc_tot", "2018", "2025")

NameError: name 'tableau_comp_1' is not defined